# Extract simulation metadata to a standalone HDF5 file — 1-peak

Extracts the `metadata` group (HDF5 attrs) from this folder's simulation file
(`ml_training_data.h5`, or `ml_training_heatmaps.h5`) and saves it into its own
small, standalone HDF5 file — for easy sharing/inspection, without carrying
around the much larger simulated spectra.

Paths below are relative to this folder — run this notebook from here, after
`ml_training_data_gen.ipynb` in the same folder has produced `simulated_data/ml_training_data.h5`.

In [1]:
import json
from pathlib import Path

import h5py


In [2]:
def save_metadata(src_filepath, dst_filepath):
    """
    Copy the 'metadata' attrs group from src_filepath into a new,
    standalone HDF5 file at dst_filepath.

    Parameters
    ----------
    src_filepath : str or Path
        Path to a trspecfit-generated HDF5 file (e.g. ml_training_data.h5
        or ml_training_heatmaps.h5) that contains a 'metadata' group.
    dst_filepath : str or Path
        Path to write the standalone metadata-only HDF5 file to.
    """
    src_filepath = Path(src_filepath)
    dst_filepath = Path(dst_filepath)

    if not src_filepath.exists():
        raise FileNotFoundError(f"Source file not found: {src_filepath}")

    with h5py.File(src_filepath, "r") as f_in:
        if "metadata" not in f_in:
            raise KeyError(f"No \'metadata\' group found in {src_filepath}")
        attrs = dict(f_in["metadata"].attrs)

    dst_filepath.parent.mkdir(parents=True, exist_ok=True)
    with h5py.File(dst_filepath, "w") as f_out:
        meta_out = f_out.create_group("metadata")
        for key, val in attrs.items():
            meta_out.attrs[key] = val
        # keep a record of where this metadata came from
        meta_out.attrs["source_file"] = str(src_filepath.resolve())

    print(f"Saved metadata: {src_filepath} -> {dst_filepath}")


In [3]:
def load_metadata(metadata_filepath, decode_json=True):
    """
    Load metadata back out of a file written by save_metadata()
    (or directly out of any trspecfit HDF5 file with a 'metadata' group).

    Parameters
    ----------
    metadata_filepath : str or Path
    decode_json : bool
        If True, json.loads() the 'parameter_space' / 'model_parameters'
        fields back into dicts (they're stored as JSON strings).

    Returns
    -------
    dict
    """
    with h5py.File(metadata_filepath, "r") as f:
        meta = dict(f["metadata"].attrs)

    if decode_json:
        for key in ("parameter_space", "model_parameters"):
            if key in meta and isinstance(meta[key], str):
                meta[key] = json.loads(meta[key])

    return meta


## Usage

In [4]:
save_metadata(
    src_filepath="simulated_data/ml_training_data.h5",
    dst_filepath="simulated_data/metadata.h5",
)


Saved metadata: simulated_data/ml_training_data.h5 -> simulated_data/metadata.h5


In [5]:
meta = load_metadata("simulated_data/metadata.h5")
meta


{'detection': 'analog',
 'dimension': np.int64(2),
 'n_configs': np.int64(100),
 'n_realizations_per_config': np.int64(30),
 'noise_level': np.float64(0.05),
 'noise_type': 'poisson',
 'parameter_space': {'GLP_01_A': {'type': 'uniform',
   'min': 8,
   'max': 12,
   'n_samples': 100},
  'GLP_01_x0': {'type': 'uniform', 'min': 8, 'max': 12, 'n_samples': 100},
  'GLP_01_x0_expFun_01_A': {'type': 'uniform',
   'min': 0.5,
   'max': 5,
   'n_samples': 100},
  'GLP_01_x0_expFun_01_tau': {'type': 'uniform',
   'min': 10,
   'max': 80,
   'n_samples': 100}},
 'seed': np.int64(42),
 'source_file': '/pscratch/sd/x/xchong/3RSE/931/simulation/time-resolved-spectroscopy-fit/examples/simulator/simulated_data/ml_training_data.h5',
 'sweep_seed': np.int64(42),
 'sweep_strategy': 'random',
 'total_datasets': np.int64(3000)}